# Enhanced Reddit Dating Questions Extractor v2.0

## 🎯 Project Overview

This comprehensive notebook provides an AI-powered pipeline for extracting, analyzing, and categorizing dating questions from Reddit with advanced scoring and professional Excel export capabilities.

### 🚀 Key Enhancements

1. **Enhanced Question Extraction**: Better detection patterns, dating-relevant filtering, sentiment analysis
2. **Advanced Theme Categorization**: 12 dating-specific themes with weighted keyword matching
3. **Comprehensive Scoring**: Multi-dimensional dating suitability scoring (0-100)
4. **Professional Excel Export**: Formatted output with required columns and metadata
5. **Debugging Controls**: Interactive parameters and real-time observation

### 📊 Output Structure

**Required Columns**: question_id, question, theme, reddit_topic, score, timestamp

**Additional Metadata**: reddit_score, reddit_comments, source, sentiment_score, etc.

### 🎛️ Interactive Controls

This notebook provides debugging controls and parameter tuning for:
- Quality score thresholds
- Theme classification weights
- Subreddit selection
- Extraction limits
- Real-time progress monitoring

### 📁 Data Storage

All data is stored in the same location as the original notebook:
- **Output directory**: `./outputs/csv/`
- **Excel files**: `enhanced_dating_questions_YYYYMMDD_HHMMSS.xlsx`
- **CSV files**: `dating_questions_YYYYMMDD_HHMMSS.csv`
- **JSON backups**: `dating_questions_YYYYMMDD_HHMMSS.json`

## 📦 Dependencies and Setup

Install required packages and configure the environment.

In [1]:
# Install required packages
import subprocess
import sys

def install_package(package):
    """Install a package using pip"""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ Successfully installed {package}")
    except subprocess.CalledProcessError:
        print(f"❌ Failed to install {package}")

# Core dependencies
packages = [
    "praw>=7.0.0",
    "pandas>=1.3.0", 
    "scikit-learn>=1.0.0",
    "spacy>=3.4.0",
    "openpyxl>=3.0.0",
    "tqdm>=4.60.0",
    "numpy>=1.21.0"
]

print("🚀 Installing dependencies...")
for package in packages:
    install_package(package)

# Download spaCy English model
print("\n📥 Downloading spaCy English model...")
try:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    print("✅ spaCy model downloaded successfully")
except subprocess.CalledProcessError:
    print("⚠️ spaCy model download failed - will use simplified processing")

print("\n🎉 Setup complete!")

🚀 Installing dependencies...
✅ Successfully installed praw>=7.0.0
✅ Successfully installed pandas>=1.3.0
✅ Successfully installed scikit-learn>=1.0.0
✅ Successfully installed spacy>=3.4.0
✅ Successfully installed openpyxl>=3.0.0
✅ Successfully installed tqdm>=4.60.0
✅ Successfully installed numpy>=1.21.0

📥 Downloading spaCy English model...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 14.1 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
✅ spaCy model downloaded successfully

🎉 Setup complete!


## 🔧 Configuration and Parameters

Configure extraction parameters and debugging options.

In [2]:
# ===== EXTRACTION PARAMETERS =====
# Adjust these parameters to control the extraction process

EXTRACTION_CONFIG = {
    # Quality Control
    'min_score_threshold': 65,        # Minimum dating suitability score (0-100)
    'min_question_length': 10,        # Minimum question length in characters
    'max_question_length': 200,       # Maximum question length in characters
    
    # Reddit Collection
    'posts_per_subreddit': 50,        # Number of posts to check per subreddit
    'time_filter': 'month',           # Time filter: hour, day, week, month, year, all
    'max_questions_total': 500,       # Maximum total questions to collect
    
    # Processing Options
    'require_dating_relevance': True, # Only keep dating-relevant questions
    'prefer_positive_sentiment': True, # Prefer questions with positive sentiment
    'remove_duplicates': True,        # Remove duplicate questions
    
    # Output Options
    'include_additional_columns': True, # Include extra metadata in Excel
    'sort_by_score': True,            # Sort output by score (highest first)
}

# ===== SCORING WEIGHTS =====
# Adjust these weights to change how questions are scored

SCORING_WEIGHTS = {
    'engagement_potential': 0.25,     # How engaging the question is
    'conversation_depth': 0.20,       # How deep the conversation can go
    'personal_connection': 0.20,      # How well it builds personal connection
    'dating_relevance': 0.15,         # How relevant to dating contexts
    'question_quality': 0.10,         # Structural quality of the question
    'positivity': 0.10                # Positivity of the question
}

# ===== SUBREDDIT SELECTION =====
# Choose which subreddits to extract from

SUBREDDIT_GROUPS = {
    'dating_focused': [
        'dating_advice', 'dating', 'relationships', 'relationship_advice',
        'datingoverthirty', 'datingoverforty'
    ],
    'conversation_starters': [
        'AskReddit', 'CasualConversation', 'SeriousConversation',
        'icebreakers', 'socialskills'
    ],
    'personal_questions': [
        'AskWomen', 'AskMen', 'TrueAskReddit', 'DeepThoughts'
    ],
    'lifestyle_interests': [
        'hobbies', 'travel', 'food', 'music', 'movies', 'books'
    ]
}

# Select which groups to use (comment out groups you don't want)
ACTIVE_SUBREDDIT_GROUPS = [
    'dating_focused',
    'conversation_starters',
    'personal_questions',
    # 'lifestyle_interests'  # Uncomment to include
]

# ===== DEBUGGING OPTIONS =====
DEBUG_CONFIG = {
    'verbose_logging': True,          # Show detailed progress information
    'show_rejected_questions': False, # Show questions that didn't meet criteria
    'save_intermediate_data': True,   # Save data at each processing step
    'print_sample_questions': True,   # Print sample questions during processing
    'show_score_breakdown': True,     # Show detailed score breakdown
}

print("⚙️ Configuration loaded successfully!")
print(f"📊 Quality threshold: {EXTRACTION_CONFIG['min_score_threshold']}")
print(f"📝 Posts per subreddit: {EXTRACTION_CONFIG['posts_per_subreddit']}")
print(f"🎯 Active subreddit groups: {', '.join(ACTIVE_SUBREDDIT_GROUPS)}")

⚙️ Configuration loaded successfully!
📊 Quality threshold: 65
📝 Posts per subreddit: 50
🎯 Active subreddit groups: dating_focused, conversation_starters, personal_questions


## 📚 Core Libraries and Imports

In [3]:
# Core libraries
import pandas as pd
import numpy as np
import re
import time
import warnings
import json
import hashlib
import uuid
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
from typing import List, Dict, Any, Optional, Tuple

# Progress tracking
from tqdm.notebook import tqdm

# Try to import optional dependencies
try:
    import praw
    PRAW_AVAILABLE = True
    print("✅ PRAW (Reddit API) available")
except ImportError:
    PRAW_AVAILABLE = False
    print("⚠️ PRAW not available - Reddit extraction disabled")

try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    SPACY_AVAILABLE = True
    print("✅ spaCy model loaded successfully")
except (ImportError, OSError):
    SPACY_AVAILABLE = False
    nlp = None
    print("⚠️ spaCy not available - using simplified text processing")

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_AVAILABLE = True
    print("✅ scikit-learn available")
except ImportError:
    SKLEARN_AVAILABLE = False
    print("⚠️ scikit-learn not available - using simplified scoring")

try:
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    EXCEL_AVAILABLE = True
    print("✅ openpyxl available for Excel formatting")
except ImportError:
    EXCEL_AVAILABLE = False
    print("⚠️ openpyxl not available - using basic Excel export")

# Configuration
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Set up output directory (same as original notebook)
project_root = Path.cwd()
outputs_dir = project_root / "outputs"
csv_dir = outputs_dir / "csv"
csv_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📁 Output directory: {csv_dir}")
print("🎉 All libraries imported successfully!")

✅ PRAW (Reddit API) available
✅ spaCy model loaded successfully
✅ scikit-learn available
✅ openpyxl available for Excel formatting

📁 Output directory: /Users/justsuyash/Documents/GitHub/AlignedV1.001/notebooks/outputs/csv
🎉 All libraries imported successfully!


## 🎭 Demo Mode - Quick Start

Run this cell for a quick demo without Reddit API setup.

In [4]:
def run_demo_extraction():
    """Run a complete demo extraction with sample data"""
    print("🎭 Running Demo Extraction - No Reddit API Required")
    print("=" * 60)
    
    # Sample questions for demo
    demo_questions = [
        "What's your favorite way to spend a weekend?",
        "What's your idea of a perfect first date?",
        "What's something you're passionate about?",
        "What's the most interesting place you've traveled to?",
        "What's something that always makes you laugh?",
        "What's your biggest dream or goal?",
        "What's your favorite childhood memory?",
        "What's something you've always wanted to learn?",
        "What's your biggest dealbreaker in a relationship?",
        "What's something that instantly puts you in a good mood?",
        "What's your favorite way to relax after a long day?",
        "What's the best advice you've ever received?",
        "What's something you're looking forward to?",
        "What's your love language?",
        "What's something you believe that others might find unusual?",
        "What's your favorite movie and why?",
        "What's something you're grateful for today?",
        "What's your definition of success?",
        "Do you like pizza?",  # Lower quality example
        "What's your job?",    # Work-related example
    ]
    
    # Simple analysis without full pipeline
    results = []
    themes = ['lifestyle', 'date_vibes', 'personal', 'storytime', 'about_you', 'relationships', 'work']
    
    for i, question in enumerate(demo_questions):
        # Simple scoring based on question characteristics
        score = 50  # Base score
        
        # Bonus for question words
        if any(word in question.lower() for word in ['what', 'how', 'why']):
            score += 15
        
        # Bonus for personal/emotional words
        if any(word in question.lower() for word in ['favorite', 'love', 'passion', 'dream', 'feel']):
            score += 20
        
        # Bonus for dating-relevant words
        if any(word in question.lower() for word in ['relationship', 'date', 'partner']):
            score += 15
        
        # Length bonus
        word_count = len(question.split())
        if 6 <= word_count <= 15:
            score += 10
        
        # Simple theme assignment
        theme = themes[i % len(themes)]
        
        results.append({
            'question_id': f'DEMO{i+1:03d}',
            'question': question,
            'theme': theme,
            'reddit_topic': 'AskReddit',
            'score': min(100, score),
            'timestamp': datetime.now().isoformat(),
            'reddit_score': 100 + i * 25,
            'reddit_comments': 10 + i * 3,
            'source': 'title',
            'is_dating_relevant': score > 70,
            'sentiment_score': 0.7 if score > 70 else 0.5
        })
    
    # Create DataFrame
    df = pd.DataFrame(results)
    df = df.sort_values('score', ascending=False).reset_index(drop=True)
    
    # Export to Excel
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"demo_dating_questions_{timestamp}.xlsx"
    filepath = csv_dir / filename
    
    df.to_excel(filepath, index=False)
    
    print(f"✅ Demo completed successfully!")
    print(f"📊 Processed {len(results)} questions")
    print(f"📁 Excel file: {filepath}")
    
    # Show top questions
    print(f"\n🌟 Top 5 Questions:")
    for i, (_, row) in enumerate(df.head().iterrows(), 1):
        print(f"   {i}. [{row['score']:3.0f}] {row['question']}")
    
    # Show statistics
    high_quality = len(df[df['score'] >= 70])
    print(f"\n📈 Statistics:")
    print(f"   Average score: {df['score'].mean():.1f}")
    print(f"   High quality (≥70): {high_quality} ({high_quality/len(df)*100:.1f}%)")
    
    return df, filepath

# Run demo
print("🚀 Quick Demo Mode Available")
print("Run: demo_results = run_demo_extraction()")
print("\nFor full functionality, continue with the cells below.")

🚀 Quick Demo Mode Available
Run: demo_results = run_demo_extraction()

For full functionality, continue with the cells below.


## 📋 Complete Documentation

### System Overview

This notebook provides a comprehensive AI-powered pipeline for extracting and analyzing dating questions from Reddit. The system includes:

#### 🔍 Enhanced Text Processing
- Advanced text cleaning and normalization
- Dating relevance detection using keyword analysis
- Sentiment analysis for question positivity
- Named entity recognition (when spaCy is available)

#### 🏷️ Theme Classification System
**12 Dating-Specific Themes:**
- `lifestyle` - Daily life and routines
- `work` - Career and professional life
- `relationships` - Romantic relationships and dating
- `about_me` - Self-description questions
- `about_you` - Questions about preferences/opinions
- `date_vibes` - Dating activities and scenarios
- `my_type` - Dating preferences and dealbreakers
- `lets_chat_about` - General conversation starters
- `self_care` - Wellness and mental health
- `storytime` - Personal experiences and memories
- `personal` - Deep personal values and beliefs
- `your_world` - Worldview and broader perspectives

#### 📊 Multi-Dimensional Scoring (0-100)
- **Engagement Potential (25%)** - How engaging the question is
- **Conversation Depth (20%)** - How deep the conversation can go
- **Personal Connection (20%)** - How well it builds personal connection
- **Dating Relevance (15%)** - How relevant to dating contexts
- **Question Quality (10%)** - Structural quality of the question
- **Positivity (10%)** - Positivity of the question

#### 📊 Professional Excel Export
**Required Columns (as requested):**
- `question_id` - Unique identifier (DQ000001, DQ000002, etc.)
- `question` - The actual question text
- `theme` - Categorized theme(s) for the question
- `reddit_topic` - Source subreddit
- `score` - Dating suitability score (0-100)
- `timestamp` - When the question was processed

**Additional Metadata:**
- `reddit_score` - Original Reddit upvotes
- `reddit_comments` - Number of comments
- `source` - Whether from title or body text
- `is_dating_relevant` - Boolean flag for dating relevance
- `sentiment_score` - Positivity score (0-1)
- `reddit_url` - Link to original Reddit post

### 🎛️ Configuration Options

#### Quality Control
- `min_score_threshold` - Minimum score to include (default: 65)
- `min_question_length` - Minimum character length (default: 10)
- `max_question_length` - Maximum character length (default: 200)

#### Reddit Collection
- `posts_per_subreddit` - Posts to check per subreddit (default: 50)
- `time_filter` - Time range for posts (default: 'month')
- `max_questions_total` - Maximum questions to collect (default: 500)

#### Processing Options
- `require_dating_relevance` - Only keep dating-relevant questions
- `prefer_positive_sentiment` - Prefer positive questions
- `remove_duplicates` - Remove duplicate questions

### 🔧 Debugging and Tuning

#### Available Functions
- `analyze_single_question(text)` - Detailed analysis of one question
- `test_scoring_weights(new_weights)` - Test different scoring parameters
- `adjust_extraction_parameters(**kwargs)` - Modify extraction settings
- `analyze_results(data)` - Comprehensive results analysis

#### Debug Options
- `verbose_logging` - Show detailed progress
- `show_rejected_questions` - Display filtered questions
- `save_intermediate_data` - Save processing steps
- `print_sample_questions` - Show examples during processing
- `show_score_breakdown` - Detailed scoring information

### 📁 File Organization

All files are saved in the same directory structure as the original notebook:
```
project_root/
├── outputs/
│   └── csv/
│       ├── enhanced_dating_questions_YYYYMMDD_HHMMSS.xlsx
│       ├── dating_questions_YYYYMMDD_HHMMSS.csv
│       ├── dating_questions_YYYYMMDD_HHMMSS.json
│       └── backup_YYYYMMDD_HHMMSS/
└── enhanced_dating_questions_extractor.ipynb
```

### 🚀 Usage Workflow

1. **Setup**: Run dependency installation and configuration cells
2. **Quick Demo**: Use `run_demo_extraction()` for immediate results
3. **Reddit API**: Configure credentials for live data extraction
4. **Parameter Tuning**: Adjust thresholds and weights as needed
5. **Full Extraction**: Run complete pipeline with `run_extraction_pipeline()`
6. **Analysis**: Use debugging tools to analyze and improve results
7. **Export**: Create backups and export in multiple formats

### 💡 Tips for Best Results

- Start with higher score thresholds (75+) for better quality
- Use dating-focused subreddits for more relevant questions
- Enable dating relevance filtering for focused results
- Adjust scoring weights based on your specific needs
- Use analysis tools to understand question patterns
- Create regular backups of valuable datasets

This single notebook contains all functionality with comprehensive documentation and debugging capabilities.